# Module 07 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_softmax_buggy.py](./adversarial_softmax_buggy.py) has two
defects:

```python
e = np.exp(logits)                      # (1) no max-subtraction: exp can overflow
return e / e.sum(axis=0, keepdims=True) # (2) axis=0 normalises across the BATCH
```

Softmax must turn each **row** (one event's logits) into a probability
distribution over classes, so the normalisation sum must run **across the
classes** (`axis=1`). Summing over `axis=0` divides each column by a batch-wide
total, so the rows no longer sum to 1 and the per-row argmax — the predicted
class — can change.

The output still has the right shape `(batch, classes)` and every value is in
[0,1], which is exactly why a shape check or an eyeball misses it.

## The fix

```python
shifted = logits - logits.max(axis=1, keepdims=True)   # overflow-safe
e = np.exp(shifted)
return e / e.sum(axis=1, keepdims=True)                 # reduce across classes
```

Subtracting the per-row max makes `exp` overflow-safe (a real concern on the GPU
with large logits) without changing the result. The corrected version, plus a
CuPy GPU drop-in, is in [softmax_solution.py](solutions/softmax_solution.py).

## Why the verification matters

The reliable test is the softmax **invariant**: every row must sum to 1. Adding an
element-wise comparison to a reference catches the axis error immediately. That is
what [../verify_softmax.py](./verify_softmax.py) does; a shape check alone would
not.

## Mapping to the 5-phase loop

- **PREDICT:** flag the reduction axis and the missing max-subtraction from
  reading the code.
- **VERIFY:** check row sums and compare to a reference, not just the shape.
- **DIAGNOSE:** on the GPU this is a per-row reduction; confirm a generated
  CUDA/CuPy version reduces over the class dimension and is overflow-safe.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!python solutions/softmax_solution.py